# SMART Units Staging ETL
### Transformation of Subcategory Asset Partitions into SMART-Compliant `units` Staging Master
---
This notebook executes Step 4.0 of the AMS-to-SMART migration:
1. Loads subcategory partitions generated from legacy `tb_assets_list`.
2. Formulates strictly unique SMART unit numbers (`{asset_number}-{subcategory_code}-{ORG}-PTRE-{YY}`).
3. Derives status (`Tersedia`, `Borrowed`, `Tidak Aktif`) and condition (`Bagus`, `Rusak`, `QC Passed`, `Hilang`, `Lelang/Hibah`).
4. Resolves location foreign keys with deterministic fallbacks.
5. Maps project numbers to `tb_project` master IDs.
6. Relocates deleted typo entry (Row 8886) to permanently deleted assets.
7. Exports structured per-subcategory unit CSVs and a consolidated `units.csv` master.
8. Executes comprehensive database integrity assertions.


In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------
# Dynamic Path Configurations
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path("/home/jovyan/data"),
        Path("../data"),
        Path("data"),
        Path("/home/ran/Devs/ams-smart-migration/data")
    ]
    for p in candidates:
        if p.exists() and (p / "ams").exists() and (p / "smart").exists():
            return p.resolve()
    return Path("data").resolve()

DATA_DIR = get_data_dir()
AMS_DIR = DATA_DIR / "ams"
SMART_DIR = DATA_DIR / "smart"
ASSETS_STAGING_DIR = SMART_DIR / "staging" / "assets"
UNITS_STAGING_DIR = SMART_DIR / "staging" / "units"
LOCATION_STAGING_DIR = SMART_DIR / "staging" / "location"
SUBCATEGORY_STAGING_DIR = SMART_DIR / "staging" / "subcategory"

UNITS_STAGING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data Directory:          {DATA_DIR}")
print(f"Assets Staging Dir:      {ASSETS_STAGING_DIR}")
print(f"Units Staging Output:    {UNITS_STAGING_DIR}")


Data Directory:          /home/jovyan/data
Assets Staging Dir:      /home/jovyan/data/smart/staging/assets
Units Staging Output:    /home/jovyan/data/smart/staging/units


## 2. Load Reference Master Datasets
We load:
1. `locations.csv`: Target location IDs mapped to legacy `ams_code`.
2. `tb_project_202610071108.csv`: Project master table mapping `no_project` to `id_project`.
3. `subcategories.csv`: Master subcategories list.
4. `_split_manifest.json`: Asset partition manifest.


In [2]:
# 1. Locations Lookup
loc_file = LOCATION_STAGING_DIR / "locations.csv"
df_loc = pd.read_csv(loc_file, dtype=str)
loc_code_to_id = dict(zip(df_loc["ams_code"], df_loc["id"]))

# Fallback mappings for unmapped legacy codes
LOC_FALLBACKS = {
    "121001": "77",  # NODATA (Graha RE Luar / ICT Pool)
    "101030": "38",  # Lantai Mezzanine (Graha RE 1)
    "102004": "39",  # Lantai 1 (Graha RE 1)
    "102011": "39",  # Lantai 1 (Graha RE 1)
}

# 2. Project Lookup
proj_file = sorted(AMS_DIR.glob("tb_project_*.csv"))[-1]
df_proj = pd.read_csv(proj_file, dtype=str)
df_proj["no_project"] = df_proj["no_project"].str.strip()
proj_no_to_id = dict(zip(df_proj["no_project"], df_proj["id_project"]))

# 3. Subcategories Master
sub_file = SUBCATEGORY_STAGING_DIR / "subcategories.csv"
df_sub = pd.read_csv(sub_file, dtype=str)
valid_sub_codes = set(df_sub["code"].unique())

# 4. Partition Manifest
manifest_file = ASSETS_STAGING_DIR / "_split_manifest.json"
with open(manifest_file, mode="r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Loaded {len(df_loc)} locations, {len(df_proj)} projects, {len(df_sub)} subcategories.")
print(f"Manifest contains {len(manifest['subcategories'])} subcategory partitions.")


Loaded 171 locations, 2748 projects, 111 subcategories.
Manifest contains 111 subcategory partitions.


## 3. Helper Functions Specification
We implement helper functions:
* **`extract_asset_end` & `generate_smart_number`**: Formulates `{asset_num}-{subcategory_code}-{ORG}-PTRE-{YY}`.
  - `ORG` is strictly `CFS` or `ICT`.
  - `YY` is strictly the last two digits of the year (months like `07`, `04` are excluded).
* **`delete_asset`**: Checks if asset is marked as permanently deleted (`deletion == 1`) and excludes it from processing.
* **`generate_status_and_condition`**: Computes status (`Tersedia`, `Dipinjam`, `Tidak Aktif`) and condition (`Bagus`, `Rusak`, `QC Passed`, `Hilang`, `Lelang/Hibah`), overriding status to `Tidak Aktif` when lost or auctioned.
* **`resolve_location_id`**: Matches `ams_code` against `locations.csv` with hierarchical fallbacks.
* **`resolve_project_id`**: Looks up `project_no` in `tb_project`.
* **`resolve_burden` & `resolve_classification`**: Maps boolean legacy flags to SMART domain values.
* **`resolve_images`**: Prepares `image_url` and staging `ams_image_url`.
* **`resolve_timestamps`**: Preserves creation and deletion dates.

In [3]:
def extract_asset_end(row):
    code = str(row.get("code", "")).strip()
    org_col = str(row.get("organizer", "")).strip()
    
    # ORG is strictly CFS or ICT
    if org_col in ("CFS", "ICT"):
        org = org_col
    elif "ICT" in code:
        org = "ICT"
    else:
        org = "CFS"
        
    # YY is strictly the last two digits of the year
    last2 = code[-2:]
    if last2.isdigit():
        yy = last2
    else:
        buy_d = str(row.get("buy_date", "")).strip()
        create_d = str(row.get("create_date", "")).strip()
        if len(buy_d) >= 4 and buy_d[:4].isdigit():
            yy = buy_d[:4][-2:]
        elif len(create_d) >= 4 and create_d[:4].isdigit():
            yy = create_d[:4][-2:]
        else:
            yy = "00"
            
    return f"-{org}-PTRE-{yy}"

def generate_smart_number(row, subcategory_code):
    code = str(row.get("code", "")).strip()
    asset_num = code.split("-")[0].strip()
    asset_end = extract_asset_end(row)
    return f"{asset_num}-{subcategory_code}{asset_end}"

def delete_asset(row):
    raw_del = row.get("deletion")
    if pd.isna(raw_del) or str(raw_del).strip().lower() in ("", "none", "nan", "null"):
        return False
    return str(raw_del).strip() == "1"

def generate_status_and_condition(row):
    raw_pem = row.get("peminjaman")
    if pd.isna(raw_pem) or str(raw_pem).strip().lower() in ("", "none", "nan", "null"):
        pem = "0"
    else:
        pem = str(raw_pem).strip()
        
    raw_stat = row.get("status")
    if pd.isna(raw_stat) or str(raw_stat).strip().lower() in ("", "none", "nan", "null"):
        stat = "0"
    else:
        stat = str(raw_stat).strip()
        
    raw_kond = row.get("kondisi")
    if pd.isna(raw_kond) or str(raw_kond).strip().lower() in ("", "none", "nan", "null"):
        kond = "0"
    else:
        kond = str(raw_kond).strip()
    
    cond_map = {
        "0": "Bagus",
        "1": "Rusak",
        "2": "QC Passed",
        "3": "Hilang",
        "4": "Lelang/Hibah"
    }
    condition = cond_map.get(kond, "Bagus")
    
    if pem == "1" and stat == "0":
        status = "Dipinjam"
    elif pem == "0" and stat == "0":
        status = "Tersedia"
    else:
        status = "Tidak Aktif"
        
    if condition in ("Hilang", "Lelang/Hibah"):
        status = "Tidak Aktif"
        
    return status, condition

def resolve_location_id(loc_val):
    if pd.isna(loc_val) or not str(loc_val).strip():
        return "77"
    val = str(loc_val).strip()
    if val in loc_code_to_id:
        return loc_code_to_id[val]
    if val in LOC_FALLBACKS:
        return LOC_FALLBACKS[val]
    return "77"

def resolve_project_id(proj_val):
    if pd.isna(proj_val) or not str(proj_val).strip():
        return None
    val = str(proj_val).strip()
    return proj_no_to_id.get(val, None)

def resolve_burden(row):
    val = str(row.get("pembebanan", "0")).strip()
    return "Project" if val == "1" else "Corporate"

def resolve_classification(row):
    val = str(row.get("assets_or_inventaris", "0")).strip()
    return "Aset" if val == "0" else "Inventaris"

def resolve_price(row):
    val = row.get("value", None)
    if pd.isna(val) or str(val).strip() == "":
        return None
    try:
        return str(float(str(val).strip()))
    except:
        return None

def resolve_images(row):
    photo = row.get("sample_photo", None)
    patch = row.get("patch", None)
    if pd.isna(photo) or not str(photo).strip():
        return None, None
    photo_str = str(photo).strip()
    image_url = f"inventory/ams/inventoryPhoto/{photo_str}"
    if pd.notna(patch) and str(patch).strip():
        patch_str = str(patch).strip().lstrip("/")
        ams_image_url = f"inventory/ams/{patch_str}{photo_str}"
    else:
        ams_image_url = f"inventory/ams/{photo_str}"
    return image_url, ams_image_url

def resolve_timestamps(row):
    create_d = row.get("create_date", None)
    del_d = row.get("deletion_date", None)
    del_val = str(row.get("deletion", "0")).strip()
    
    created_at = str(create_d).strip() if pd.notna(create_d) else None
    if del_val == "1" and pd.notna(del_d) and str(del_d).strip():
        updated_at = str(del_d).strip()
    else:
        updated_at = created_at
    return created_at, updated_at

## 4. Deletion Typo Relocation (Row 8886)
Row ID `8886` (`03427-ACC-LS-CFS-PTRE-0724`, `deletion = 1`) was a mistaken entry in AMS where the admin typed `-0724`, deleted it, and re-entered it as row ID `8891` (`03427-ACC-LS-CFS-PTRE-24`, `deletion = 0`).

Per Case 1 (*admin realizes mistake and deletes record*), we relocate row 8886 into `permanently_deleted_assets.csv` and remove it from `ACC-LS.csv`, guaranteeing 100% uniqueness of generated codes across active assets.


In [4]:
acc_ls_path = ASSETS_STAGING_DIR / "ACC-LS.csv"
perm_del_path = ASSETS_STAGING_DIR / "permanently_deleted_assets.csv"

df_acc_ls = pd.read_csv(acc_ls_path, dtype=str)
df_perm_del = pd.read_csv(perm_del_path, dtype=str)

row_8886 = df_acc_ls[df_acc_ls["id"] == "8886"]
if len(row_8886) > 0:
    print("Relocating row 8886 from ACC-LS to permanently_deleted_assets...")
    df_perm_del = pd.concat([df_perm_del, row_8886], ignore_index=True)
    df_perm_del.to_csv(perm_del_path, index=False)
    
    df_acc_ls = df_acc_ls[df_acc_ls["id"] != "8886"].copy()
    df_acc_ls.to_csv(acc_ls_path, index=False)
    
    # Update manifest count
    manifest["subcategories"]["ACC-LS"]["row_count"] = len(df_acc_ls)
    manifest["total_partitioned_assets"] = sum(s["row_count"] for s in manifest["subcategories"].values())
    manifest["total_permanently_deleted_assets"] = len(df_perm_del)
    with open(manifest_file, mode="w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Row 8886 relocated. ACC-LS active rows: {len(df_acc_ls)}, Perm deleted rows: {len(df_perm_del)}")
else:
    print("Row 8886 is already relocated.")


Row 8886 is already relocated.


## 5. Subcategory Partition Processing Loop
We iterate across all 111 subcategories in `manifest`, apply all transformations, and write:
1. `data/smart/staging/units/<subcategory_code>.csv`
2. An accumulated in-memory record collection with sequential `id` (PK) values (`1` .. `6943`).


In [5]:
units_manifest = {}
all_units_records = []
all_deleted_records = []
current_unit_id = 1

for sub_code, meta in sorted(manifest["subcategories"].items()):
    asset_file = ASSETS_STAGING_DIR / meta["file"]
    unit_out_file = UNITS_STAGING_DIR / f"{sub_code}.csv"
    
    if not asset_file.exists() or meta["row_count"] == 0:
        # Empty partition
        empty_unit_df = pd.DataFrame(columns=[
            "id", "ams_asset_id", "number", "legacy_number", "subcategory_code",
            "location_id", "status", "condition", "price", "image_url",
            "ams_image_url", "type", "classification", "burden", "project_id",
            "created_at", "updated_at"
        ])
        empty_unit_df.to_csv(unit_out_file, index=False)
        units_manifest[sub_code] = {
            "file": f"{sub_code}.csv",
            "row_count": 0
        }
        continue
        
    df_sub_assets = pd.read_csv(asset_file, dtype=str)
    sub_units = []
    
    for _, r in df_sub_assets.iterrows():
        if delete_asset(r):
            all_deleted_records.append(r)
            continue
            
        smart_num = generate_smart_number(r, sub_code)
        st, cd = generate_status_and_condition(r)
        loc_id = resolve_location_id(r.get("location"))
        proj_id = resolve_project_id(r.get("project_no"))
        price = resolve_price(r)
        img_url, ams_img_url = resolve_images(r)
        rent_type = str(r.get("rent_type", "LT")).strip() or "LT"
        classification = resolve_classification(r)
        burden = resolve_burden(r)
        created_at, updated_at = resolve_timestamps(r)
        
        unit_row = {
            "id": current_unit_id,
            "ams_asset_id": str(r["id"]).strip(),
            "number": smart_num,
            "legacy_number": str(r["code"]).strip(),
            "subcategory_code": sub_code,
            "location_id": loc_id,
            "status": st,
            "condition": cd,
            "price": price,
            "image_url": img_url,
            "ams_image_url": ams_img_url,
            "type": rent_type,
            "classification": classification,
            "burden": burden,
            "project_id": proj_id,
            "created_at": created_at,
            "updated_at": updated_at
        }
        sub_units.append(unit_row)
        all_units_records.append(unit_row)
        current_unit_id += 1
        
    df_sub_units = pd.DataFrame(sub_units)
    df_sub_units.to_csv(unit_out_file, index=False)
    units_manifest[sub_code] = {
        "file": f"{sub_code}.csv",
        "row_count": len(df_sub_units)
    }

print(f"Transformed {len(all_units_records):,} units across {len(units_manifest)} subcategories.")
print(f"Permanently excluded {len(all_deleted_records):,} deleted assets (deletion == 1).")

# Update permanently_deleted_assets.csv for audit trail
perm_del_path = ASSETS_STAGING_DIR / "permanently_deleted_assets.csv"
if perm_del_path.exists() and all_deleted_records:
    df_perm_del = pd.read_csv(perm_del_path, dtype=str)
    df_new_del = pd.DataFrame(all_deleted_records)
    cols = [c for c in df_perm_del.columns if c in df_new_del.columns]
    combined_perm_del = pd.concat([df_perm_del, df_new_del[cols]], ignore_index=True).drop_duplicates(subset=["id"])
    combined_perm_del.to_csv(perm_del_path, index=False)
    print(f"Audit: permanently_deleted_assets.csv now contains {len(combined_perm_del):,} records.")

Transformed 6,515 units across 111 subcategories.
Permanently excluded 424 deleted assets (deletion == 1).
Audit: permanently_deleted_assets.csv now contains 443 records.


## 6. Consolidated Master Export & Manifest
Export the consolidated `units.csv` master file and `_units_manifest.json`.


In [6]:
df_units_master = pd.DataFrame(all_units_records)
master_units_path = UNITS_STAGING_DIR / "units.csv"
df_units_master.to_csv(master_units_path, index=False)
print(f"Saved consolidated units master to: {master_units_path.name} ({len(df_units_master):,} rows)")

# Units Manifest
units_manifest_path = UNITS_STAGING_DIR / "_units_manifest.json"
manifest_payload = {
    "generated_at": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ"),
    "total_units": len(df_units_master),
    "total_subcategories": len(units_manifest),
    "status_distribution": df_units_master["status"].value_counts().to_dict(),
    "condition_distribution": df_units_master["condition"].value_counts().to_dict(),
    "burden_distribution": df_units_master["burden"].value_counts().to_dict(),
    "classification_distribution": df_units_master["classification"].value_counts().to_dict(),
    "subcategories": units_manifest
}
with open(units_manifest_path, mode="w", encoding="utf-8") as f:
    json.dump(manifest_payload, f, indent=2)
print(f"Saved units manifest to: {units_manifest_path.name}")


Saved consolidated units master to: units.csv (6,515 rows)
Saved units manifest to: _units_manifest.json


## 7. Data Quality & Referential Integrity Assertions
We execute automated assertions verifying:
1. Exact total count (6,943 units).
2. Continuous sequential `id` PK (1 to 6,943).
3. 100% unique `number` (0 collisions).
4. Suffix format conforms strictly to `-ORG-PTRE-YY`.
5. 0 nulls in required columns.
6. 100% of `location_id` exist in `locations.csv`.
7. 100% of `burden == 'Project'` have non-null `project_id`.


In [7]:
# 1. Total row count assertion
assert len(df_units_master) == 6515, f"Expected 6,515 rows, got {len(df_units_master)}"

# 2. Sequential unique PK assertion
assert df_units_master["id"].is_monotonic_increasing, "IDs must be strictly monotonic increasing"
assert df_units_master["id"].min() == 1, "Min ID must be 1"
assert df_units_master["id"].max() == 6515, f"Max ID must be 6,515, got {df_units_master['id'].max()}"
assert df_units_master["id"].nunique() == 6515, "All IDs must be unique"

# 3. Code uniqueness assertion
assert df_units_master["number"].nunique() == 6515, f"Duplicate unit numbers detected! Unique count: {df_units_master['number'].nunique()}"

# 4. Suffix pattern assertion (-ORG-PTRE-YY)
suffix_pattern = r"^.*-(CFS|ICT)-PTRE-\d{2}$"
invalid_suffixes = df_units_master[~df_units_master["number"].str.match(suffix_pattern)]
assert len(invalid_suffixes) == 0, f"Invalid code suffix found: {invalid_suffixes['number'].tolist()[:5]}"

# 5. Non-null required columns
required_cols = ["id", "ams_asset_id", "number", "location_id", "status", "condition", "type", "classification", "burden", "created_at", "updated_at"]
for col in required_cols:
    null_count = df_units_master[col].isna().sum()
    assert null_count == 0, f"Column {col} has {null_count} null values!"

# 6. Valid location foreign keys
valid_loc_ids = set(df_loc["id"].unique())
invalid_locs = df_units_master[~df_units_master["location_id"].isin(valid_loc_ids)]
assert len(invalid_locs) == 0, f"Invalid location IDs found: {invalid_locs['location_id'].unique()}"

# 7. Project foreign key mapping for Project burden
project_units = df_units_master[df_units_master["burden"] == "Project"]
assert len(project_units) == 92, f"Expected 92 Project burden units, got {len(project_units)}"
assert project_units["project_id"].notna().all(), "All Project burden units must have a non-null project_id"

# 8. Domain constraints
assert set(df_units_master["status"].unique()).issubset({"Tersedia", "Dipinjam", "Tidak Aktif"}), "Invalid status detected"
assert set(df_units_master["condition"].unique()).issubset({"Bagus", "Rusak", "QC Passed", "Hilang", "Lelang/Hibah"}), "Invalid condition detected"
assert set(df_units_master["type"].unique()).issubset({"LT", "ST"}), "Invalid type detected"
assert set(df_units_master["classification"].unique()).issubset({"Aset", "Inventaris"}), "Invalid classification detected"

print("="*55)
print("ALL 8 INTEGRITY ASSERTIONS PASSED SUCCESSFULLY!")
print(f"  • Total Units Processed:        {len(df_units_master):,}")
print(f"  • Total Subcategories Exported: {len(units_manifest)}")
print(f"  • Unique Unit Numbers:          {df_units_master['number'].nunique():,} (0 duplicates)")
print(f"  • Status Distribution:          {df_units_master['status'].value_counts().to_dict()}")
print(f"  • Condition Distribution:       {df_units_master['condition'].value_counts().to_dict()}")
print(f"  • Classification Distribution:  {df_units_master['classification'].value_counts().to_dict()}")
print(f"  • Burden Distribution:          {df_units_master['burden'].value_counts().to_dict()}")
print("="*55)

ALL 8 INTEGRITY ASSERTIONS PASSED SUCCESSFULLY!
  • Total Units Processed:        6,515
  • Total Subcategories Exported: 111
  • Unique Unit Numbers:          6,515 (0 duplicates)
  • Status Distribution:          {'Tersedia': 3491, 'Tidak Aktif': 2027, 'Dipinjam': 997}
  • Condition Distribution:       {'Bagus': 4615, 'Rusak': 1470, 'QC Passed': 357, 'Lelang/Hibah': 69, 'Hilang': 4}
  • Classification Distribution:  {'Aset': 5606, 'Inventaris': 909}
  • Burden Distribution:          {'Corporate': 6423, 'Project': 92}
